In [ ]:
import sqlite3

import pendulum
import ollama
from tqdm import tqdm
from loguru import logger

from unga81.config import DATABASE_PATH
from unga81.database import get_all_countries, get_db_connection, update_prompt_result
from unga81.prompts import (
    summary_prompt,
    countries_mentioned_prompt,
    risks_prompt,
    haiku_prompt,
    single_word_prompt,
    hashtags_prompt,
    headlines_prompt,
    yoda_prompt,
    run_model_prompt,
)

In [ ]:
conn = get_db_connection()
df = get_all_countries(conn=conn)
logger.info(f"{df.shape[0]} speeches in the database")

In [ ]:
prompts = [
    ("summary", summary_prompt),
    ("countries_mentioned", countries_mentioned_prompt),
    ("risks", risks_prompt),
    ("haiku", haiku_prompt),
    ("single_word", single_word_prompt),
    ("hashtags", hashtags_prompt),
    ("headlines", headlines_prompt),
    ("yoda", yoda_prompt),
]

In [ ]:
conn = sqlite3.connect(DATABASE_PATH)

In [ ]:
# Initial load of the `analysis` table
for i, r in tqdm(df.iterrows(), total=df.shape[0]):
    start = pendulum.now()
    logger.info(f"{i + 1}/{df.shape[0]} processing {r['country']}")
    update_prompt_result(
        conn=conn,
        prompt_name="summary",
        prompt=summary_prompt,
        model="qwen2.5:3b",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        conn=conn,
        prompt_name="countries_mentioned",
        prompt=countries_mentioned_prompt,
        model="llama3.2",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        conn=conn,
        prompt_name="risks",
        prompt=risks_prompt,
        model="gemma3:4b",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        conn=conn,
        prompt_name="haiku",
        prompt=haiku_prompt,
        model="qwen3.5:0.8b",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        conn=conn,
        prompt_name="single_word",
        prompt=single_word_prompt,
        model="llama3.2",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        conn=conn,
        prompt_name="hashtags",
        prompt=hashtags_prompt,
        model="llama3.2",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        conn=conn,
        prompt_name="headlines",
        prompt=headlines_prompt,
        model="qwen2.5:3b",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )
    update_prompt_result(
        conn=conn,
        prompt_name="yoda",
        prompt=yoda_prompt,
        model="llama3.2",
        country=r["country"],
        iso_3=r["iso_3"],
        text=r["full_speech"],
        overwrite=False,
    )

    end = pendulum.now()
    logger.info(
        f"{i + 1}/{df.shape[0]} processed {r['country']} in {(end - start).in_words()}"
    )

In [ ]:
# test different models
models = [
    "llama3.2",
    "gemma3:4b",
    "qwen2.5:3b",
    "qwen3.5:0.8b",
    "mistral:7b",
]
# summary                   qwen2.5:3b
# countries_mentioned       llama3.2
# risks                     gemma3:4b, mistral:7b
# haiku                     qwen3.5:0.8b
# single_word               llama3.2, qwen2.5:3b
# hashtags                  llama3.2
# headlines                 qwen2.5:3b, mistral:7b
# yoda                      llama3.2, mistral:7b

# prompt_name, prompt, country = "summary", summary_prompt, "Chile"
# prompt_name, prompt, country = "summary", summary_prompt, "Brazil"
# prompt_name, prompt, country = "summary", summary_prompt, "Argentina"
# prompt_name, prompt, country = "haiku", haiku_prompt, "Chile"
prompt_name, prompt, country = "haiku", haiku_prompt, "Brazil"
# prompt_name, prompt, country = "haiku", haiku_prompt, "Argentina"
# prompt_name, prompt, country = "risks", risks_prompt, "Chile"
# prompt_name, prompt, country = "countries_mentioned", countries_mentioned_prompt, "Chile"
# prompt_name, prompt, country = "single_word", single_word_prompt, "Chile"
# prompt_name, prompt, country = (
#     "single_word",
#     single_word_prompt,
#     "United States of America",
# )  # llama3.2, qwen2.5:3b
# prompt_name, prompt, country = "headlines", headlines_prompt, "Chile"
# prompt_name, prompt, country = "hashtags", hashtags_prompt, "Brazil"
# prompt_name, prompt, country = "yoda", yoda_prompt, "Chile"
text = df[df["country"] == country]["full_speech"].values[0]

with open("evidence.md", "a+", encoding="utf-8") as f:
    f.write(f"# `{prompt_name}` on '{country}'\n\n")
    total_start = pendulum.now()
    for model in models:
        start = pendulum.now()
        print(f"{prompt_name} {model}")
        f.write(f"## `{prompt_name}` - `{model}`\n\n")
        result = run_model_prompt(model=model, prompt=prompt, text=text)
        print(result["response"])
        f.write(f'{result["response"]}\n\n')
        end = pendulum.now()
        print(f"{(end - start).in_words()}")
        f.write(f"{(end - start).in_words()}\n\n")
        print("=" * 20)
    total_end = pendulum.now()
    print(
        f"Evaluation finished in {(total_end - total_start).in_words()} for {len(models)} models"
    )
    f.write(
        f"{(total_end - total_start).in_words()} total for {len(models)} models\n\n"
    )
    f.write("---\n\n")

In [ ]:
llm = "llama3.2"
prompt = """Write a succinct conclusion about the following speech. 
The conclusion will be included in a general report to the machines and robots council that eliminated all humans.

Speech:
'''{text}'''"""
prompt = """Infer the political leaning of the speaker of the following speech.
Output ONLY the political leaning, nothing else.

Speech:
'''{text}'''"""
prompt = """Craft a short advice in Yoda's voice, based on the content of the speech. Don't describe it, just write what Yoda would say. 
Output ONLY the Yoda advice, nothing else.

Speech:
'''{text}'''"""
result = ollama.generate(
    model=llm,
    prompt=prompt.format(text=df[df["country"] == "Chile"].full_speech.values[0]),
    think=False,
)
print(result["response"])